# Fruit & Vegetable Classification: Model Development + Streamlit Deployment (Google Colab)

Everything in one notebook:
1. Download and extract the dataset
2. Build, train and evaluate the CNN (same model as the class notebook)
3. Save the model
4. Create the Streamlit app and open it with a public link

**Use a GPU** (*Runtime > Change runtime type > T4 GPU*) and run the cells from top to bottom.

## 1. Imports

In [ ]:
import os, json, zipfile
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers
from tensorflow.keras.models import Sequential

print("TensorFlow:", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))

## 2. Download and extract the dataset

In [ ]:
%pip install -q gdown streamlit
import gdown

ZIP_PATH = "Fruits_Vegetables.zip"
if not os.path.exists(ZIP_PATH):
    gdown.download(id="1CGiAWso43GCsNo_faRq4jdDIlmwy7YI4", output=ZIP_PATH, quiet=False)

if not os.path.exists(ZIP_PATH):
    raise SystemExit("Download failed. Upload Fruits_Vegetables.zip using the Files panel on the left, then re-run this cell.")

with zipfile.ZipFile(ZIP_PATH) as z:
    z.extractall("Fruits_Veg_Dataset")
print("Extracted.")

## 3. Find the train / validation / test folders and remove broken images

In [ ]:
from PIL import Image

IMG_EXT = (".jpg", ".jpeg", ".png", ".bmp", ".webp")

# remove unreadable images so training does not crash
removed = 0
for root, _, files in os.walk("Fruits_Veg_Dataset"):
    for f in files:
        p = os.path.join(root, f)
        if f.startswith("._") or f.lower() in (".ds_store", "thumbs.db"):
            os.remove(p)
        elif f.lower().endswith(IMG_EXT):
            try:
                with Image.open(p) as im:
                    im.verify()
            except Exception:
                os.remove(p); removed += 1
print("Broken images removed:", removed)

# locate the three split folders
dirs = {}
for root, subdirs, _ in os.walk("Fruits_Veg_Dataset"):
    if "__MACOSX" in root:
        continue
    name = os.path.basename(root).lower()
    if len(subdirs) >= 2:
        if name == "train": dirs["train"] = root
        elif name in ("validation", "valid", "val"): dirs["val"] = root
        elif name == "test": dirs["test"] = root
print(dirs)
assert {"train", "val", "test"} <= set(dirs), "Could not find train/validation/test folders: set the paths by hand in this cell."

## 4. Load the data

In [ ]:
img_height = 180
img_width = 180
BATCH = 32

def load(path, shuffle):
    return tf.keras.utils.image_dataset_from_directory(
        path, image_size=(img_height, img_width), batch_size=BATCH, shuffle=shuffle, seed=42)

train_images = load(dirs["train"], True)
val_images = load(dirs["val"], False)
test_images = load(dirs["test"], False)

data_labels = train_images.class_names
print(len(data_labels), "classes:", data_labels)

plt.figure(figsize=(10, 10))
for images, labels in train_images.take(1):
    for i in range(9):
        plt.subplot(3, 3, i + 1)
        plt.imshow(images[i].numpy().astype("uint8"))
        plt.title(data_labels[labels[i]])
        plt.axis("off")
plt.show()

AUTOTUNE = tf.data.AUTOTUNE
train_images = train_images.prefetch(AUTOTUNE)
val_images = val_images.prefetch(AUTOTUNE)
test_images = test_images.prefetch(AUTOTUNE)

## 5. Build the CNN
Same architecture as the class notebook. One fix: `Rescaling(1/255)` instead of `Rescaling(.1/255)`
(the original scaled pixels 10x too small, which hurts learning).

In [ ]:
model = Sequential([
    layers.Rescaling(1.0 / 255),
    layers.Conv2D(16, 3, padding="same", activation="relu"),
    layers.MaxPooling2D(),
    layers.Conv2D(32, 3, padding="same", activation="relu"),
    layers.MaxPooling2D(),
    layers.Conv2D(64, 3, padding="same", activation="relu"),
    layers.MaxPooling2D(),
    layers.Dropout(0.2),
    layers.Flatten(),
    layers.Dense(128, activation="relu"),
    layers.Dense(len(data_labels)),   # logits
])

model.compile(optimizer="adam",
              loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
              metrics=["accuracy"])

## 6. Train

In [ ]:
history = model.fit(train_images, validation_data=val_images, epochs=10)
model.summary()

## 7. Training curves and test accuracy

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
ax[0].plot(history.history["accuracy"], label="train")
ax[0].plot(history.history["val_accuracy"], label="val")
ax[0].set_title("Accuracy"); ax[0].legend()
ax[1].plot(history.history["loss"], label="train")
ax[1].plot(history.history["val_loss"], label="val")
ax[1].set_title("Loss"); ax[1].legend()
plt.show()

test_loss, test_acc = model.evaluate(test_images, verbose=0)
print(f"Test accuracy: {test_acc:.4f}")

## 8. Save the model and class names

In [ ]:
model.save("Fruite_and_veg_model.keras")
with open("class_names.json", "w") as f:
    json.dump(data_labels, f)
print("Saved Fruite_and_veg_model.keras and class_names.json")

## 9. Test a prediction inside the notebook

In [ ]:
def predictor(img_path):
    img = tf.keras.utils.load_img(img_path, target_size=(img_height, img_width))
    arr = tf.expand_dims(tf.keras.utils.img_to_array(img), 0)
    score = tf.nn.softmax(model.predict(arr, verbose=0)[0]).numpy()
    return data_labels[int(np.argmax(score))], float(np.max(score) * 100)

# pick one image from the test set
for images, labels in test_images.take(1):
    path_img = images[0].numpy().astype("uint8")
    plt.imshow(path_img); plt.axis("off")
    score = tf.nn.softmax(model.predict(images[:1], verbose=0)[0]).numpy()
    plt.title(f"Predicted: {data_labels[int(np.argmax(score))]} ({np.max(score)*100:.1f}%) | True: {data_labels[labels[0]]}")
    plt.show()

## 10. Create the Streamlit app (`app.py`)

In [ ]:
%%writefile app.py
"""Streamlit app: Fruit & Vegetable classifier.

Run with:  streamlit run app.py
Needs Fruite_and_veg_model.keras and class_names.json in the same folder.
"""
import json
import os

import numpy as np
import pandas as pd
import streamlit as st
import tensorflow as tf
from PIL import Image

MODEL_PATH = "Fruite_and_veg_model.keras"
IMG_HEIGHT = 180
IMG_WIDTH = 180

with open("class_names.json") as f:
    CLASS_NAMES = json.load(f)

st.set_page_config(page_title="Fruit & Veg Classifier", page_icon="🥦")


@st.cache_resource(show_spinner="Loading model...")
def load_model(path: str):
    return tf.keras.models.load_model(path)


def predict(model, image: Image.Image):
    """Same steps as the notebook: resize to 180x180, raw 0-255 pixels
    (the model has its own Rescaling layer), softmax on the logits."""
    img = image.convert("RGB").resize((IMG_WIDTH, IMG_HEIGHT), Image.BILINEAR)
    arr = np.expand_dims(np.array(img, dtype="float32"), axis=0)
    logits = model.predict(arr, verbose=0)
    score = tf.nn.softmax(logits[0]).numpy()
    return score


st.title("🥦🍎 Fruit & Vegetable Classifier")
st.write("Upload a photo (or take one) and the CNN will tell you which of the "
         f"{len(CLASS_NAMES)} fruits/vegetables it is.")

# --- model file ---
if not os.path.exists(MODEL_PATH):
    st.warning(f"`{MODEL_PATH}` was not found next to app.py. Upload it here:")
    up_model = st.file_uploader("Model file (.keras)", type=["keras"], key="model")
    if up_model is not None:
        with open(MODEL_PATH, "wb") as f:
            f.write(up_model.getbuffer())
        st.rerun()
    st.stop()

model = load_model(MODEL_PATH)

# --- image input ---
tab_upload, tab_camera = st.tabs(["Upload image", "Use camera"])
with tab_upload:
    file = st.file_uploader("Choose an image", type=["jpg", "jpeg", "png", "webp"])
with tab_camera:
    cam = st.camera_input("Take a photo")

source = file or cam
if source is not None:
    image = Image.open(source)
    score = predict(model, image)
    top = np.argsort(score)[::-1][:5]

    col1, col2 = st.columns(2)
    col1.image(image, caption="Your image", use_container_width=True)
    col2.metric("Prediction", CLASS_NAMES[top[0]].title())
    col2.write(f"Confidence: **{score[top[0]] * 100:.2f}%**")
    if score[top[0]] < 0.5:
        col2.info("Low confidence: try a clearer photo with one item and a plain background.")

    st.subheader("Top 5 predictions")
    df = pd.DataFrame({"Confidence (%)": [score[i] * 100 for i in top]},
                      index=[CLASS_NAMES[i] for i in top])
    st.bar_chart(df)

## 11. Run the app and get a public link
Colab runs on a remote machine, so `localhost` can't be opened directly.
This cell starts Streamlit plus a free Cloudflare tunnel. Click the printed `trycloudflare.com` link.

In [ ]:
import re, subprocess, sys, time

if not os.path.exists("cloudflared"):
    subprocess.run(["wget", "-q",
                    "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
                    "-O", "cloudflared"], check=True)
    os.chmod("cloudflared", 0o755)

st_proc = subprocess.Popen([sys.executable, "-m", "streamlit", "run", "app.py",
                            "--server.port", "8501", "--server.headless", "true"])
time.sleep(8)

cf_log = open("cf.log", "w")
cf_proc = subprocess.Popen(["./cloudflared", "tunnel", "--url", "http://localhost:8501"],
                           stdout=cf_log, stderr=subprocess.STDOUT)

public_url = None
for _ in range(30):
    time.sleep(2)
    m_ = re.search(r"https://[-a-z0-9]+\.trycloudflare\.com", open("cf.log").read())
    if m_:
        public_url = m_.group(0)
        break

print("Open your app here:", public_url if public_url else "not ready yet, run: print(open('cf.log').read())")

## 12. Stop the app when finished

In [ ]:
cf_proc.terminate()
st_proc.terminate()
print("Stopped.")